This cell ensures that the necessary `pub-dialogue` package is installed, either from a local environment or directly from GitHub if running in a Colab environment. This step is crucial for enabling subsequent analyses of the extracted text data, as the package likely contains functions and tools specifically designed for processing and evaluating the quality of public dialogue text related to science and technology.

In [ ]:
# @title Install pub-dialogue package
# Local: run 'pip install -e .' once in the repo root.
# Colab:  installs directly from GitHub.
try:
    import pub_dialogue
except ImportError:
    import subprocess, sys
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "git+https://github.com/mlatcl/pub-dialogue.git",
    ])
    import pub_dialogue

This cell sets up the necessary libraries and configurations for analyzing paragraph-level text extracted from public dialogue PDFs on science and technology. By defining parameters such as random seeds for reproducibility, model specifications for embeddings and clustering, and processing settings for text chunking, it establishes a foundation for subsequent data quality assessments and ensures that the analysis can handle variations in document structure effectively. This initial setup is crucial for maintaining consistency and accuracy throughout the analysis process, particularly when dealing with diverse and potentially unstructured text data.

In [ ]:
# @title Import libraries and define configuration
# NOTE: constants below are pre-import defaults; cell 6 overrides them from AccessStage.
import random
import warnings
from pathlib import Path

import numpy as np

# Reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Chunking quality thresholds — overridden by AccessStage in cell 6
MIN_CHUNK_WORDS = 40
MAX_CHUNK_WORDS = 500
MIN_CHUNK_CHARS = 80

# Sentence-fallback chunking parameters (used by access.extract_chunks_from_pdf)
SENTENCE_FALLBACK_TARGET_WORDS = 300
SENTENCE_FALLBACK_MIN_PARAGRAPHS = 3

# Paths — detect Colab vs local (overridden by AccessStage in cell 6)
try:
    import google.colab  # noqa: F401
    _REPO_ROOT = Path('/content')
except ImportError:
    _REPO_ROOT = Path('.')
OUTPUT_FOLDER     = _REPO_ROOT / 'outputs'
CHECKPOINT_FOLDER = _REPO_ROOT / 'checkpoints'

for folder in [OUTPUT_FOLDER, CHECKPOINT_FOLDER]:
    folder.mkdir(exist_ok=True)

warnings.filterwarnings('ignore')

print("Configuration loaded (assess-phase only — no API key required)")


This cell initializes the necessary modules and utility functions from the `pub_dialogue` library, which are essential for assessing the quality of the extracted text from public dialogue PDFs. By importing various functions for data validation, diagnostics, and text processing, it sets the groundwork for subsequent analyses, ensuring that the data can be effectively evaluated and any issues can be addressed systematically. This step is crucial for maintaining the integrity of the analysis, as it directly impacts the reliability of insights drawn from the dialogue data.

In [ ]:
# @title Load pub_dialogue — assess-phase helpers only
# This notebook is pure assess: no API key, no extraction prompts, no clustering.
try:
    import pub_dialogue.access as access
    import pub_dialogue.assess as assess
    from pub_dialogue.utils import (
        show_status, show_complete, show_warning,
    AccessStage, AssessStage,
        filter_missing_source_text,
        is_privacy_text,
        vocabulary_frequency_diagnostic,
        extract_chunks_from_pdf, reset_chunk_stats, get_chunk_stats,
        MIN_CHUNK_WORDS, MIN_CHUNK_CHARS, MAX_CHUNK_WORDS,
        SENTENCE_FALLBACK_TARGET_WORDS, SENTENCE_FALLBACK_MIN_PARAGRAPHS,
    )
    print("pub_dialogue imported successfully")
except ImportError as _e:
    print(f"pub_dialogue not found: {_e}")
    raise

In [ ]:
# @title CIP-0010: Create stage objects and derive path constants
# Must run after pub_dialogue import (cell above).
_access  = AccessStage()
_assess  = AssessStage(access=_access)

OUTPUT_FOLDER     = _access.output_folder
CHECKPOINT_FOLDER = _access.checkpoint_folder
PDF_FOLDER        = _access.pdf_folder
MIN_CHUNK_WORDS   = _access.min_chunk_words
MAX_CHUNK_WORDS   = _access.max_chunk_words
MIN_CHUNK_CHARS   = _access.min_chunk_chars

print(f'OUTPUT_FOLDER={OUTPUT_FOLDER}  CHECKPOINT_FOLDER={CHECKPOINT_FOLDER}')


## Load pre-extracted chunks

This notebook operates on `paragraph_chunks.csv` written by `01_processing.ipynb`.
No API key is required — all cells here are question-agnostic quality checks
(Fynesse **assess** phase).

In [ ]:
import pandas as pd

chunks_df = pd.read_csv(OUTPUT_FOLDER / "paragraph_chunks.csv")
print(f"Loaded {len(chunks_df):,} chunks from paragraph_chunks.csv")


This code snippet sets up the visualization environment by importing the Matplotlib library and configuring its style and resolution for clarity in subsequent plots. This is crucial for effectively visualizing the data quality assessment results, as clear and aesthetically pleasing graphics will enhance the interpretability of the findings related to the extracted text from the public dialogue documents.

In [ ]:
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 150

This cell generates a visual summary of the data quality for the paragraph-level text extracted from the 66 UK public dialogue PDF documents. By assessing and plotting the quality metrics, it helps identify potential issues in the text data, such as inconsistencies or gaps, which is crucial for ensuring the reliability of subsequent analyses in understanding public perceptions of science and technology.

In [ ]:
show_status("Running data quality summary...")

This cell produces a visual representation of the data quality for the extracted paragraph-level text, allowing researchers to quickly identify potential issues such as missing data or inconsistencies across the 66 UK public dialogue documents. By summarizing these quality metrics, it facilitates informed decisions on data cleaning and preprocessing, which are crucial steps in ensuring the reliability of subsequent analyses in the study of public dialogue on science and technology.

In [ ]:
# @title Summarise paragraph-level data quality
_assess.plot_quality(chunks_df)
plt.show()

This cell evaluates the quality of text chunks extracted from the public dialogue documents by identifying those that may be incorrectly classified as meaningful content, such as bibliography entries or survey tables. By reporting the contamination rate based on specified thresholds for word and character counts, it provides critical insights into the reliability of the data, which is essential for ensuring the integrity of subsequent analyses in the study of public dialogue on science and technology.

In [ ]:
# @title (v16) Chunk content-quality diagnostic
# Reports the proportion of kept chunks that look like bibliography fragments
# or survey-table rows, so the analyst can see the contamination rate at the
# chosen MIN_CHUNK_WORDS floor.

show_status("Running chunk content-quality diagnostic...")
chunks_df = assess.flag_chunk_quality(chunks_df, OUTPUT_FOLDER, MIN_CHUNK_WORDS, MIN_CHUNK_CHARS)
show_complete("Chunk quality diagnostic complete — see chunk_quality_flagged.csv")